In [4]:
import asyncio
import json
import subprocess
import ollama
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

In [13]:
# ── Config ───────────────────────────────────────────────────────────────────
MODEL         = "qwen3:14b"          # any model pulled in Ollama
OLLAMA_HOST   = "http://127.0.0.1:11434"  # default Ollama address
ALLOWED_DIR   = "/home/jovyan"       # directory the MCP server can access
MCP_CRML_URL = "https://crml-mcp.narancsle.cc/mcp"   # change to your actual MCP server
# ─────────────────────────────────────────────────────────────────────────────

# Point the ollama client at your local server
client = ollama.AsyncClient(host=OLLAMA_HOST)
print("Config ready.")

Config ready.


In [8]:
# Cell 2 — Verify Ollama is up and the model is available
import httpx

async def check_ollama():
    async with httpx.AsyncClient() as http:
        r = await http.get(f"{OLLAMA_HOST}/api/tags")
        models = [m["name"] for m in r.json().get("models", [])]
        print(f"Ollama is up. Available models: {models}")
        if not any(MODEL in m for m in models):
            print(f"\n⚠ Model '{MODEL}' not found. Run: ollama pull {MODEL}")
        else:
            print(f"✓ Model '{MODEL}' is ready.")

await check_ollama()

Ollama is up. Available models: ['qwen3:14b']
✓ Model 'qwen3:14b' is ready.


In [9]:
# Cell 3 — Core agent logic (works with any stdio MCP server)

async def get_tools(session: ClientSession):
    """Fetch MCP tools and convert to Ollama format."""
    mcp_tools = await session.list_tools()
    ollama_tools = [
        {
            "type": "function",
            "function": {
                "name": t.name,
                "description": t.description,
                "parameters": t.inputSchema,
            },
        }
        for t in mcp_tools.tools
    ]
    return mcp_tools.tools, ollama_tools


async def agent_loop(session: ClientSession, user_message: str, verbose: bool = True):
    """Run a single user turn, calling tools as needed until the model stops."""
    mcp_tools, ollama_tools = await get_tools(session)
    messages = [{"role": "user", "content": user_message}]

    if verbose:
        print(f"{'='*60}")
        print(f"User: {user_message}")
        print(f"Tools available: {[t.name for t in mcp_tools]}\n")

    while True:
        response = await client.chat(
            model=MODEL,
            messages=messages,
            tools=ollama_tools,
            options={"temperature": 0.6},
        )

        msg = response.message

        # No tool calls → final answer
        if not msg.tool_calls:
            if verbose:
                print(f"Assistant: {msg.content}")
            return msg.content

        # Append the assistant turn (with tool calls)
        messages.append(msg)

        # Execute each tool via MCP
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = tool_call.function.arguments
            if isinstance(args, str):
                args = json.loads(args)

            if verbose:
                print(f"→ Tool call: '{name}' args={args}")

            result = await session.call_tool(name, args)
            result_text = str(result.content)

            if verbose:
                preview = result_text[:300] + ("..." if len(result_text) > 300 else "")
                print(f"← Result: {preview}\n")

            messages.append({"role": "tool", "content": result_text})


print("Agent logic defined.")

Agent logic defined.


In [ ]:
# Cell 4 — Run the agent with the filesystem MCP server
#
# mcp-server-filesystem exposes these tools:
#   read_file, write_file, list_directory, create_directory,
#   move_file, search_files, get_file_info
#
# The server is launched as a subprocess — no separate terminal needed.

server_params = StdioServerParameters(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", ALLOWED_DIR],
    env=None,
)

async def run(user_message: str):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await agent_loop(session, user_message)

# ── Ask the model something that requires using filesystem tools ──────────────
# await run("List the files in my home directory, then read the first .ipynb file you find and summarise what it does.")

In [18]:
from mcp.client.streamable_http import streamablehttp_client
from mcp import ClientSession

async def run_http(user_message: str):
    async with streamablehttp_client(MCP_CRML_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await agent_loop(session, user_message)

await run_http("What tools do you have available?")

User: What tools do you have available?
Tools available: ['get_coding_instructions', 'check_syntax', 'list_CRML_resource_files', 'read_hint_file', 'search_hints']

Assistant: I have several tools available to help with CRML (Computational Resource Modeling Language):

1. **get_coding_instructions** - Provides general guidance on using CRML language features
2. **check_syntax** - Validates CRML model syntax and identifies errors with line/column numbers
3. **list_CRML_resource_files** - Lists all available language resource files (e.g., type definitions, syntax guides)
4. **read_hint_file** - Retrieves content from specific CRML resource files (requires filename from list_CRML_resource_files first)
5. **search_hints** - Searches for keywords across all CRML resource files to find relevant language features

Would you like me to demonstrate any of these tools?


'I have several tools available to help with CRML (Computational Resource Modeling Language):\n\n1. **get_coding_instructions** - Provides general guidance on using CRML language features\n2. **check_syntax** - Validates CRML model syntax and identifies errors with line/column numbers\n3. **list_CRML_resource_files** - Lists all available language resource files (e.g., type definitions, syntax guides)\n4. **read_hint_file** - Retrieves content from specific CRML resource files (requires filename from list_CRML_resource_files first)\n5. **search_hints** - Searches for keywords across all CRML resource files to find relevant language features\n\nWould you like me to demonstrate any of these tools?'

In [16]:
await run_http("Write a simple CRML model for a DC/DC converter.")

User: Write a simple CRML model for a DC/DC converter.
Tools available: ['get_coding_instructions', 'check_syntax', 'list_CRML_resource_files', 'read_hint_file', 'search_hints']

→ Tool call: 'get_coding_instructions' args={}
← Result: [TextContent(type='text', text='# CRML — High-Level Coding Instructions\n\nCRML (Contract Requirements Modeling Language) is a declarative language that aims to formalize requirements and connect their evaluation to simulation models of physical systems.\nIt is preferred to have some isolation betwe...

Assistant: Here's a simple CRML model for a DC/DC converter:

```crml
model DCDCConverter is {
    // External system variables
    Real inputVoltage is external;        // Input DC voltage
    Boolean switchState is external;      // Switch position (ON/OFF)
    Real inductorCurrent is external;     // Inductor current
    Real outputVoltage is external;       // Output voltage
    Real capacitorVoltage is external;    // Capacitor voltage

    // Requir

"Here's a simple CRML model for a DC/DC converter:\n\n```crml\nmodel DCDCConverter is {\n    // External system variables\n    Real inputVoltage is external;        // Input DC voltage\n    Boolean switchState is external;      // Switch position (ON/OFF)\n    Real inductorCurrent is external;     // Inductor current\n    Real outputVoltage is external;       // Output voltage\n    Real capacitorVoltage is external;    // Capacitor voltage\n\n    // Requirement: Output voltage must stay above 5V\n    Boolean voltageRequirement is during (new Clock (outputVoltage >= 5)) ensure outputVoltage >= 5;\n    \n    // Requirement: Inductor current must not exceed 2A\n    Boolean currentRequirement is during (new Clock (inductorCurrent <= 2)) ensure inductorCurrent <= 2;\n};\n```\n\nThis model:\n1. Declares external variables representing key converter parameters\n2. Defines two safety requirements using CRML's temporal operators:\n   - Voltage requirement: Ensures output voltage remains above 5